In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
afsadasasdasdas_computer_vision_project_dataset_path = kagglehub.dataset_download('afsadasasdasdas/computer-vision-project-dataset')

print('Data source import complete.')


100%|██████████| 2.27G/2.27G [00:23<00:00, 103MB/s]

Extracting files...


Data source import complete.


In [ ]:
import kagglehub

#get csv file with embeddings
bert_embeddings_dataset_path = kagglehub.dataset_download("tesnimejemmazi/embeddings-1")

print("Path to dataset files:", bert_embeddings_dataset_path)

Path to dataset files: /kaggle/input/embeddings-1


REGRESSION

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel
from PIL import Image, UnidentifiedImageError 
from PIL import ImageFile 
ImageFile.LOAD_TRUNCATED_IMAGES = True 
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import joblib


# === Configuration ===
poster_images_base = os.path.join(afsadasasdasdas_computer_vision_project_dataset_path, "poster_images")

if os.path.exists(os.path.join(poster_images_base, "poster_images", "poster_images")):
    image_dir = os.path.join(poster_images_base, "poster_images", "poster_images")
elif os.path.exists(os.path.join(poster_images_base, "poster_images")):
    image_dir = os.path.join(poster_images_base, "poster_images")
else:
    image_dir = poster_images_base
print(f"Image directory set to: {image_dir}")


csv_path = os.path.join(bert_embeddings_dataset_path, "titles_with_bert_embeddings_1.csv")
print(f"CSV path set to: {csv_path}")

SCORE_COL = "imdb_score"

# === Load Data ===
try:
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded CSV from: {csv_path}")
except FileNotFoundError:
    print(f"ERROR: CSV file not found at {csv_path}.")
    print("Double-check the path within the 'bert_embeddings_dataset_path' and ensure the dataset handle is correct.")
    raise

required_cols = ['ocr_title', 'bert_cls_embedding', 'image_path', SCORE_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing required columns for regression: {missing_cols}. Please ensure '{SCORE_COL}' is present.")

df = df.dropna(subset=required_cols)
df['bert_cls_embedding'] = df['bert_cls_embedding'].apply(eval)
df[SCORE_COL] = pd.to_numeric(df[SCORE_COL], errors='coerce')
df = df.dropna(subset=[SCORE_COL])
print(f"Data type of '{SCORE_COL}' after preprocessing: {df[SCORE_COL].dtype}")

df['image_path'] = df['image_path'].apply(lambda x: x.replace("poster_images/", ""))
df['image_path'] = df['image_path'].apply(lambda x: x.replace("poster_images/poster_images/", ""))


# === Dataset Class ===
class OCRPosterDataset(Dataset):
    def __init__(self, dataframe, image_dir, score_col, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.score_col = score_col
        self.transform = transform
        self.original_len = len(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        
        #limit the number of retries for a given index.
        max_retries = 10
        retries = 0

        while retries < max_retries:
            row = self.data.iloc[idx]
            rel_path = row['image_path']
            image_path = os.path.join(self.image_dir, rel_path)

            try:
                image = Image.open(image_path).convert('RGB')
                if self.transform:
                    image = self.transform(image)
                text_feat = torch.tensor(row['bert_cls_embedding'], dtype=torch.float32)
                target_score = torch.tensor(row[self.score_col], dtype=torch.float32)
                return image, text_feat, target_score
            except (FileNotFoundError, UnidentifiedImageError, OSError) as e: #error
                print(f"Error processing image {image_path} (CSV row: {row['image_path']}): {e}")
                #move to the next index
                idx = (idx + 1) % self.original_len #cycle back to start if at end

                #if we've cycled through all items or hit max retries, something is wrong
                if retries == self.original_len -1 or retries == max_retries -1:
                    raise RuntimeError("Could not find a valid image after multiple attempts. Dataset might be empty or severely corrupted.")
                retries += 1
        
        raise RuntimeError(f"Failed to load image after {max_retries} retries for index {idx}. Original error: {e}")

# === Transforms ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# === Train/Val Split ===
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_ds = OCRPosterDataset(train_df, image_dir, SCORE_COL, transform)
val_ds = OCRPosterDataset(val_df, image_dir, SCORE_COL, transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

# === Model ===
class ViT_OCR_Model(nn.Module):
    def __init__(self, vit_model_name='google/vit-base-patch16-224', text_feat_dim=768, output_dim=1):
        super(ViT_OCR_Model, self).__init__()
        self.vit = ViTModel.from_pretrained(vit_model_name)
        self.vit_head = nn.Linear(self.vit.config.hidden_size, 256)
        self.text_fc = nn.Linear(text_feat_dim, 256)
        self.regressor = nn.Linear(512, output_dim)

    def forward(self, image, text_feat):
        vit_out = self.vit(pixel_values=image).last_hidden_state[:, 0, :]
        vit_feat = self.vit_head(vit_out)
        text_feat = self.text_fc(text_feat)
        combined = torch.cat([vit_feat, text_feat], dim=1)
        return self.regressor(combined).squeeze(1)

# === Train & Evaluate ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViT_OCR_Model(output_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.MSELoss()

def train(model, loader):
    model.train()
    total_loss = 0
    for images, text_feats, targets in tqdm(loader):
        images = images.to(device)
        text_feats = text_feats.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images, text_feats)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

best_mae = float("inf")
best_model_path = "best-vit-ocr-regression-model.pth"

def evaluate(model, loader, save_best=False):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, text_feats, targets in loader:
            images = images.to(device)
            text_feats = text_feats.to(device)
            targets = targets.to(device)

            outputs = model(images, text_feats)
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    mae = mean_absolute_error(all_targets, all_preds)
    mse = mean_squared_error(all_targets, all_preds)

    print(f"MAE: {mae:.3f} | MSE: {mse:.3f}")

    global best_mae
    if save_best and mae < best_mae:
        best_mae = mae
        torch.save(model.state_dict(), best_model_path)
        print(f"Best checkpoint saved to: {best_model_path}")

    return mae, mse

def save_checkpoint(model, epoch, path="checkpoint_epoch_{:02d}.pt"):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, path.format(epoch))

# === Main Loop ===
for epoch in range(3):
    loss = train(model, train_loader)
    print(f"Epoch {epoch+1} Loss: {loss:.4f}")
    evaluate(model, val_loader, save_best=True)
    save_checkpoint(model, epoch+1)

# === Final Evaluation ===
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    mae, mse = evaluate(model, val_loader)
else:
    print(f"Warning: {best_model_path} not found. Skipping final evaluation with best model.")
    mae, mse = evaluate(model, val_loader, save_best=False)

print("\n Sample predictions:")
model.eval()
try:
    sample_imgs, sample_texts, sample_targets = next(iter(val_loader))
    with torch.no_grad():
        preds = model(sample_imgs.to(device), sample_texts.to(device))
        top_preds = preds.cpu().numpy()

    for i in range(min(5, len(sample_targets))):
        print(f" True: {sample_targets[i].item():.2f} →  Pred: {top_preds[i].item():.2f}")
except StopIteration:
    print("Validation loader is empty. Cannot generate sample predictions.")
except RuntimeError as e:
    print(f"Could not get sample predictions due to a DataLoader error: {e}")


print(f"\nFinal Metrics:\nMAE: {mae} MSE: {mse}")

Image directory set to: /root/.cache/kagglehub/datasets/afsadasasdasdas/computer-vision-project-dataset/versions/4/poster_images/poster_images/poster_images
CSV path set to: /kaggle/input/embeddings-1/titles_with_bert_embeddings_1.csv
Successfully loaded CSV from: /kaggle/input/embeddings-1/titles_with_bert_embeddings_1.csv
Data type of 'imdb_score' after preprocessing: float64


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 739/739 [16:05<00:00,  1.31s/it]


Epoch 1 Loss: 5.9322
MAE: 1.740 | MSE: 5.201
Best checkpoint saved to: best-vit-ocr-regression-model.pth


100%|██████████| 739/739 [16:02<00:00,  1.30s/it]


Epoch 2 Loss: 4.7463
MAE: 1.699 | MSE: 5.322
Best checkpoint saved to: best-vit-ocr-regression-model.pth


100%|██████████| 739/739 [16:06<00:00,  1.31s/it]


Epoch 3 Loss: 3.3120
MAE: 1.760 | MSE: 5.757
MAE: 1.699 | MSE: 5.322

 Sample predictions:
 True: 3.00 →  Pred: 3.71
 True: 8.70 →  Pred: 5.36
 True: 5.40 →  Pred: 3.75
 True: 5.00 →  Pred: 5.35
 True: 1.00 →  Pred: 3.36

Final Metrics:
MAE: 1.6994577466413023 MSE: 5.321593132303361
